In [6]:
from google.colab import files, drive
import os, glob

# 1) Se não existir no /content, faça upload
if not os.path.exists('/content/df_merge_clean.parquet'):
    up = files.upload()  # selecione df_merge_clean.parquet
    # após o upload, ajuste o nome se vier diferente
    for k in up.keys():
        if k != 'df_merge_clean.parquet':
            os.rename(k, 'df_merge_clean.parquet')

# 2) Confirme
!ls -lh /content | grep -i parquet
PATH = '/content/df_merge_clean.parquet'

Saving df_merge_clean.parquet to df_merge_clean.parquet
-rw-r--r-- 1 root root 178M Sep 23 03:07 df_merge_clean.parquet


In [7]:
# S-last ROBUST — median blend + cap p85*1.02 + gamma 0.98  (TARGET_SHARE≈0.951)
import duckdb, os
from google.colab import files

con = duckdb.connect()
PATH = "/content/df_merge_clean.parquet"  # já está no /content
OUT  = "submission_s_last_robust.parquet"

# VIEW base (2022)
con.execute(f"CREATE OR REPLACE VIEW raw AS SELECT * FROM read_parquet('{PATH}')")
con.execute("""
CREATE OR REPLACE VIEW base AS
SELECT premise, categoria, pdv, produto,
       CAST(iso_week AS INT) AS iso_week,
       SUM(quantity) AS qty
FROM raw
WHERE iso_year=2022
GROUP BY 1,2,3,4,5
""")

TARGET_SHARE = 0.951
CAP_Q, CAP_K = 0.85, 1.02
GAMMA = 0.98

sql = f"""
WITH recent AS (
  SELECT premise,categoria,pdv,produto,
         SUM(qty) FILTER (WHERE iso_week BETWEEN 45 AND 52) AS qty_recent
  FROM base GROUP BY 1,2,3,4
),
ranked AS (
  SELECT *,
         SUM(qty_recent) OVER (PARTITION BY premise,categoria) AS grp_sum,
         SUM(qty_recent) OVER (PARTITION BY premise,categoria
           ORDER BY qty_recent DESC
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS csum
  FROM recent
),
cut AS (
  SELECT premise,categoria,pdv,produto
  FROM ranked
  WHERE qty_recent>0 AND csum/NULLIF(grp_sum,0) <= {TARGET_SHARE}
),
weekly AS (
  SELECT b.premise,b.categoria,b.pdv,b.produto,b.iso_week, SUM(qty) qty
  FROM base b JOIN cut c USING(premise,categoria,pdv,produto)
  GROUP BY 1,2,3,4,5
),
wmean AS (
  SELECT premise,categoria,pdv,produto,
         (0.55*AVG(qy) FILTER (WHERE iso_week=52)
        + 0.30*AVG(qy) FILTER (WHERE iso_week=51)
        + 0.10*AVG(qy) FILTER (WHERE iso_week=50)
        + 0.05*AVG(qy) FILTER (WHERE iso_week=49)) AS seed_wdecay,
         AVG(qy) FILTER (WHERE iso_week BETWEEN 49 AND 52)     AS seed_mean4
  FROM (SELECT premise,categoria,pdv,produto,iso_week,qty AS qy FROM weekly)
  GROUP BY 1,2,3,4
),
lnz8 AS (
  SELECT premise,categoria,pdv,produto, FIRST(qty) AS seed_lnz8
  FROM (
    SELECT w.*, ROW_NUMBER() OVER (PARTITION BY premise,categoria,pdv,produto
      ORDER BY iso_week DESC) rn
    FROM weekly w
    WHERE iso_week BETWEEN 45 AND 52 AND qty>0
  ) t WHERE rn=1
  GROUP BY premise,categoria,pdv,produto
),
season AS (
  SELECT premise,categoria,
         NULLIF(AVG(qty) FILTER (WHERE iso_week BETWEEN 1 AND 5),0)
         / NULLIF(AVG(qty) FILTER (WHERE iso_week BETWEEN 45 AND 52),0) AS s_jan_rel
  FROM base GROUP BY 1,2
),
seed0 AS (
  SELECT w.premise,w.categoria,w.pdv,w.produto,
         /* mediana( lnz8, mean4, wdecay ) */
         (SELECT median(x) FROM (VALUES
            (COALESCE(l.seed_lnz8, NULL)),
            (COALESCE(w.seed_mean4, NULL)),
            (COALESCE(w.seed_wdecay, NULL))
          ) AS t(x)) AS qty_seed
  FROM wmean w LEFT JOIN lnz8 l USING(premise,categoria,pdv,produto)
),
caps AS (
  SELECT premise,categoria, quantile_cont(qty_seed, {CAP_Q}) AS pcap
  FROM seed0 GROUP BY 1,2
),
seed_adj AS (
  SELECT s.pdv, s.produto,
         GREATEST(0.0, LEAST(
           {GAMMA} * s.qty_seed * COALESCE(se.s_jan_rel,1.0),
           c.pcap * {CAP_K}
         )) AS quantidade
  FROM seed0 s
  LEFT JOIN season se USING(premise,categoria)
  LEFT JOIN caps   c  USING(premise,categoria)
),
weeks AS (SELECT * FROM (VALUES (1),(2),(3),(4),(5)) t(semana))
SELECT
  CAST(semana AS INT) AS semana,
  CAST(pdv AS BIGINT) AS pdv,
  CAST(produto AS BIGINT) AS produto,
  CAST(FLOOR(quantidade) AS BIGINT) AS quantidade
FROM weeks CROSS JOIN seed_adj
"""

sub = con.execute(sql).df()
print("rows:", len(sub), "| dups:", sub.duplicated(['semana','pdv','produto']).sum())
assert len(sub) <= 1_500_000, "Linhas >1,5M. Ajuste TARGET_SHARE."
con.execute(f"COPY ({sql}) TO '{OUT}' (FORMAT 'parquet')")
files.download(OUT)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 1477000 | dups: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>